# 📊 AiStock 可视化分析

## 分析目标
- ✅ 动态价格分布
- ✅ 板块配置分析
- ✅ 风险收益分析
- ✅ 组合表现可视化

## 可视化内容
- 价格分布图
- 板块权重图
- 盈亏比分析
- 风险矩阵

In [ ]:
# 环境设置
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print("✅ 环境设置完成")

In [ ]:
# 导入系统模块
from base_services.config_service import ConfigService
from base_services.cache_service import CacheService
from base_services.database_service import DatabaseService
from dynamic_price_system.data.data_loader import DataLoader
from dynamic_price_system.core.dynamic_price_engine import DynamicPriceEngine
from config.global_settings import DATABASE_MAIN_URL, DB_POOL_CONFIG

print("✅ 系统模块导入成功")

In [ ]:
# 初始化并获取数据
config = ConfigService(system_name='dynamic_price')
cache = CacheService(max_size=1000, ttl=3600)
db = DatabaseService(DATABASE_MAIN_URL, DB_POOL_CONFIG)

data_loader = DataLoader(config, cache, db, enable_cache=True)
engine = DynamicPriceEngine(config, cache)

# 加载数据
stocks_data = data_loader.load_all_stocks()
macro_data = data_loader.load_all_macro()
financial_data = data_loader.load_all_financial()

# 计算动态价格
results = engine.calculate_all(stocks_data, financial_data, macro_data)

print(f"✅ 获取 {len(results)} 只标的动态价格数据")

## 1️⃣ 动态价格分布

In [ ]:
# 转换为 DataFrame
df_results = pd.DataFrame(results)

# 价格分布图
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 入场价分布
axes[0, 0].hist(df_results['entry_price'], bins=10, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('入场价格分布')
axes[0, 0].set_xlabel('价格 (元)')
axes[0, 0].set_ylabel('数量')
axes[0, 0].grid(True, alpha=0.3)

# 2. 盈亏比分布
axes[0, 1].hist(df_results['profit_loss_ratio'], bins=10, edgecolor='black', alpha=0.7, color='orange')
axes[0, 1].set_title('盈亏比分布')
axes[0, 1].set_xlabel('盈亏比')
axes[0, 1].set_ylabel('数量')
axes[0, 1].grid(True, alpha=0.3)

# 3. 基本面评分分布
axes[1, 0].hist(df_results['fundamental_score'], bins=10, edgecolor='black', alpha=0.7, color='green')
axes[1, 0].set_title('基本面评分分布')
axes[1, 0].set_xlabel('评分')
axes[1, 0].set_ylabel('数量')
axes[1, 0].grid(True, alpha=0.3)

# 4. 宏观系数分布
axes[1, 1].hist(df_results['macro_factor'], bins=10, edgecolor='black', alpha=0.7, color='purple')
axes[1, 1].set_title('宏观系数分布')
axes[1, 1].set_xlabel('系数')
axes[1, 1].set_ylabel('数量')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2️⃣ 板块配置分析

In [ ]:
# 板块分析
sector_stats = df_results.groupby('sector').agg({
    'code': 'count',
    'profit_loss_ratio': 'mean',
    'fundamental_score': 'mean',
    'macro_factor': 'mean'
}).rename(columns={'code': 'count'})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. 板块数量
axes[0].bar(sector_stats.index, sector_stats['count'], color='skyblue')
axes[0].set_title('各板块标的数量')
axes[0].set_xlabel('板块')
axes[0].set_ylabel('数量')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3)

# 2. 板块平均盈亏比
axes[1].bar(sector_stats.index, sector_stats['profit_loss_ratio'], color='coral')
axes[1].set_title('各板块平均盈亏比')
axes[1].set_xlabel('板块')
axes[1].set_ylabel('盈亏比')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3️⃣ 风险收益矩阵

In [ ]:
# 风险收益散点图
plt.figure(figsize=(12, 8))

# 按建议着色
color_map = {
    '强烈推荐': 'green',
    '推荐': 'blue',
    '观望': 'orange',
    '谨慎': 'red'
}

for rec in df_results['recommendation'].unique():
    subset = df_results[df_results['recommendation'] == rec]
    plt.scatter(
        subset['fundamental_score'],
        subset['profit_loss_ratio'],
        label=rec,
        color=color_map.get(rec, 'gray'),
        s=100,
        alpha=0.7,
        edgecolors='black'
    )

plt.xlabel('基本面评分', fontsize=12)
plt.ylabel('盈亏比', fontsize=12)
plt.title('风险收益矩阵', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4️⃣ 投资建议分布

In [ ]:
# 建议分布饼图
rec_counts = df_results['recommendation'].value_counts()

plt.figure(figsize=(8, 8))
colors = ['green', 'blue', 'orange', 'red']
plt.pie(
    rec_counts.values,
    labels=rec_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    explode=[0.05] * len(rec_counts)
)
plt.title('投资建议分布', fontsize=14)
plt.show()

## 5️⃣ 价格对比分析

In [ ]:
# 当前价 vs 入场价 vs 目标价
top_stocks = df_results.nlargest(10, 'profit_loss_ratio')

x = np.arange(len(top_stocks))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - width, top_stocks['current_price'], width, label='当前价')
ax.bar(x, top_stocks['entry_price'], width, label='入场价')
ax.bar(x + width, top_stocks['target_price'], width, label='目标价')

ax.set_xlabel('标的')
ax.set_ylabel('价格 (元)')
ax.set_title('Top 10 推荐标的价格对比')
ax.set_xticks(x)
ax.set_xticklabels(top_stocks['code'], rotation=45)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 📊 分析总结

In [ ]:
print("="*60)
print("📋 可视化分析总结")
print("="*60)
print(f"✅ 总标的数：{len(df_results)}只")
print(f"✅ 板块数量：{df_results['sector'].nunique()}个")
print(f"✅ 平均盈亏比：{df_results['profit_loss_ratio'].mean():.2f}")
print(f"✅ 平均基本面评分：{df_results['fundamental_score'].mean():.1f}")
print(f"✅ 强烈推荐：{(df_results['recommendation']=='强烈推荐').sum()}只")
print(f"✅ 推荐：{(df_results['recommendation']=='推荐').sum()}只")
print("="*60)

# 清理资源
db.close()
cache.clear()
print("\n✅ 资源已清理")
print("\n🎉 可视化分析完成！")